# Day 40：LangChain Agent & `@tool` 生态与安全沙箱实战

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第四阶段（Stage 4：Agent 智能体开发）第 1 天！今天我们将亲手实现 `@tool` 装饰器、`StructuredTool`、`ToolCallingAgent` 与 `AgentExecutor`，并结合 Mock Tavily 搜索、AST 安全数学求值与防路径穿越沙箱文件工具，构建可观测、可审计的企业级工具调用智能体（`v0.4.0-day40`）。

## 1. 环境初始化与导入 `project_iterations/day_40` 模块
定位仓库根目录并将 `project_iterations/day_40` 加入 `sys.path`，导入核心组件。

In [1]:
import json
import sys
import tempfile
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day40_dir = repo_root / "project_iterations" / "day_40"
if str(day40_dir) not in sys.path:
    sys.path.insert(0, str(day40_dir))

from zhishu_agent import (
    AgentExecutor,
    StructuredTool,
    ToolCallingAgent,
    create_sandbox_file_tools,
    safe_math_eval,
    tavily_search,
    tool,
)

print("已加载模块路径:", day40_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_40


## 2. `@tool` 装饰器：自动提取函数名、Docstring 与类型注解生成 JSON Schema
定义一个自定义企业工具 `query_employee`，查看 `StructuredTool` 自动生成 OpenAI Function Calling Schema 的效果。

In [2]:
@tool
def query_employee(emp_id: str, include_dept: bool = True) -> dict:
    """查询星辰数科员工基础档案.

    Args:
        emp_id: 员工工号，例如 EMP-10086。
        include_dept: 是否附带所属部门信息。
    """
    return {"emp_id": emp_id, "name": "陈默", "dept": "AI 研发部" if include_dept else None}

print("工具类型:", type(query_employee).__name__)
print(json.dumps(query_employee.to_openai_tool(), ensure_ascii=False, indent=2))
print("直接调用结果:", query_employee("EMP-10086"))

工具类型: StructuredTool
{
  "type": "function",
  "function": {
    "name": "query_employee",
    "description": "查询星辰数科员工基础档案.\n\nArgs:\n    emp_id: 员工工号，例如 EMP-10086。\n    include_dept: 是否附带所属部门信息。",
    "parameters": {
      "type": "object",
      "properties": {
        "emp_id": {
          "type": "string",
          "description": "员工工号，例如 EMP-10086。"
        },
        "include_dept": {
          "type": "boolean",
          "description": "是否附带所属部门信息。",
          "default": true
        }
      },
      "required": [
        "emp_id"
      ]
    }
  }
}
直接调用结果: {'emp_id': 'EMP-10086', 'name': '陈默', 'dept': 'AI 研发部'}


## 3. 三重安全沙箱工具验证：AST 数学求值、Mock Tavily 与路径穿越防御
验证 `safe_math_eval` 拦截 `__import__('os')` 与 `eval()`，以及 `create_sandbox_file_tools` 拦截 `../../etc/passwd`。

In [3]:
print("合法数学求值:", safe_math_eval.invoke({"expression": "(12 + 8) * 3 / 2 + sqrt(16)"}))

for bad_expr in ["__import__('os').system('id')", "eval('1+1')"]:
    try:
        safe_math_eval.invoke({"expression": bad_expr})
    except ValueError as exc:
        print(f"[AST 拦截成功] {bad_expr} -> {exc}")

with tempfile.TemporaryDirectory() as tmp:
    read_file, write_file = create_sandbox_file_tools(Path(tmp))
    print(write_file.invoke({"file_path": "notes/day40.txt", "content": "智枢 Agent 沙箱测试"}))
    print("沙箱读取:", read_file.invoke({"file_path": "notes/day40.txt"}))
    try:
        read_file.invoke({"file_path": "../../etc/passwd"})
    except PermissionError as exc:
        print(f"[路径穿越拦截成功] {exc}")

合法数学求值: 34.0
[AST 拦截成功] __import__('os').system('id') -> 安全拦截：仅允许调用白名单数学函数 (abs, round, min, max, pow, sqrt)
[AST 拦截成功] eval('1+1') -> 安全拦截：仅允许调用白名单数学函数 (abs, round, min, max, pow, sqrt)
已安全写入 13 字符至沙箱文件: notes/day40.txt
沙箱读取: 智枢 Agent 沙箱测试
[路径穿越拦截成功] 安全拦截：检测到路径穿越攻击 '../../etc/passwd'，已拒绝访问


## 4. `ToolCallingAgent` & `AgentExecutor` 多步调度与 `intermediate_steps` 审计
运行多步计算+写文件任务，以及联网搜索任务，检查返回的 `output` 与 `intermediate_steps`。

In [4]:
with tempfile.TemporaryDirectory() as tmp:
    sandbox_root = Path(tmp)
    read_file, write_file = create_sandbox_file_tools(sandbox_root)
    tools = [tavily_search, safe_math_eval, read_file, write_file]

    agent = ToolCallingAgent(tools=tools)
    executor = AgentExecutor(agent=agent, tools=tools, max_iterations=5)

    res1 = executor.invoke({"input": "搜索 LangChain Tool Calling 最佳实践"})
    print("[任务 1] output:", res1["output"][:120] + "...")
    print("[任务 1] steps:", [(a.tool, type(obs).__name__) for a, obs in res1["intermediate_steps"]])

    res2 = executor.invoke({"input": "请计算 (128 + 72) * 3 / 4 并写入 calc_result.txt"})
    print("\n[任务 2] output:", res2["output"])
    for idx, (action, obs) in enumerate(res2["intermediate_steps"], start=1):
        print(f"  Step {idx}: tool={action.tool}, input={action.tool_input} -> obs={obs}")
    print("沙箱文件内容验证:", (sandbox_root / "calc_result.txt").read_text(encoding="utf-8"))

[任务 1] output: 已完成工具 [tavily_search] 调用，结果：[{'title': 'LangChain Tool Calling & @tool 最佳实践', 'url': 'https://docs.zhishu.ai/langchain/t...
[任务 1] steps: [('tavily_search', 'list')]

[任务 2] output: 已完成工具 [write_file] 调用，结果：已安全写入 5 字符至沙箱文件: calc_result.txt
  Step 1: tool=safe_math_eval, input={'expression': '(128 + 72) * 3 / 4'} -> obs=150.0
  Step 2: tool=write_file, input={'file_path': 'calc_result.txt', 'content': '150.0'} -> obs=已安全写入 5 字符至沙箱文件: calc_result.txt
沙箱文件内容验证: 150.0
